# Export

In [2]:
def extract_marked_text(text):
    start = text.find('{')
    end = text.find('}')
    
    if start != -1 and end != -1 and start < end:
        answer = text[start + 1 : end]
        return answer.lower()
    else:
        return "No Char"

In [3]:
import pandas as pd
import json
import os
import re

# Define the words
WORDs = [
    "Syndrome",
    "question"
]

# Initialize an empty list to store DataFrames
all_data_frames = []

# Loop through each word
for WORD in WORDs:
    gt_data = pd.read_json(f"/Users/log/Github/VideoFilters/images/{WORD}/configurations_combined.json")
    # gt_data = pd.read_json(f"/Users/log/Github/BlindLLMs/revisions/CircledWordRevisions/blank_spacing/images/{WORD}/configurations_combined.json")
    # remplace ./images/ with ./images_second_prompt/
    # gt_data["image_path"] = gt_data["image_path"].apply(
    #     lambda x: x.replace("./images/", "./images_second_prompt/")
    # )
    # Generate model output file paths and read the content if the file exists
    gt_data["model-output-file"] = gt_data["image_path"].apply(
        lambda x: x.replace(".png", "") + "-claude-3-5-sonnet-20240620-output.md"
    )
    gt_data["model-output-raw"] = gt_data["model-output-file"].apply(
        lambda x: (open(x, "r").read() if os.path.exists(x) else None)
    )

    # Drop rows with missing sonnet output
    gt_data = gt_data.dropna(subset=["model-output-raw"])
    # print(len(gt_data))

    gt_data["predicted"] = gt_data["model-output-raw"].apply(extract_marked_text)
    gt_data["predicted"] = gt_data["predicted"].apply(lambda x: x.lower())

    # print(gt_data["predicted"].value_counts())
    # Calculate ground truth and correctness
    gt_data["gt"] = gt_data["circled_letter"].str.lower()
    gt_data["is_prediction_correct"] = gt_data["gt"] == gt_data["predicted"]
    gt_data["word_label"] = WORD  # Add a column to identify the word
    # print(f"Word: {WORD}, Accuracy: {gt_data['is_prediction_correct'].mean()}")
    
    no_spaces_data = gt_data[gt_data['num_spaces'] == 0]
    if not no_spaces_data.empty:
        accuracy = no_spaces_data['is_prediction_correct'].mean()
        print(f"Word: {WORD}, Accuracy (num_spaces = 0): {accuracy}")
    # Append to the list
    all_data_frames.append(gt_data)

# Concatenate all DataFrames into one
final_data_frame = pd.concat(all_data_frames, ignore_index=True)

Word: Syndrome, Accuracy (num_spaces = 0): 0.9375
Word: question, Accuracy (num_spaces = 0): 1.0


In [4]:
accuracy_by_spaces = final_data_frame.groupby(['word_label', 'num_spaces'])['is_prediction_correct'].mean().unstack()
accuracy_by_spaces


num_spaces,0,1,2,3
word_label,,,,
Syndrome,0.9375,1.0,1.0,1.0
question,1.0000,1.0,1.0,1.0


In [9]:
final_data_frame["Model"] = ["Sonnet-3"] * len(final_data_frame)

In [10]:
final_data_frame.to_pickle("./Sonnet-3-highlighted.pkl")

In [11]:
# group by word and average is_prediction_correct
final_data_frame.groupby("word_label")["is_prediction_correct"].mean()

word_label
Acknowledgement         0.958333
Subdermatoglyphic       0.813725
tHyUiKaRbNqWeOpXcZvM    0.706250
Name: is_prediction_correct, dtype: float64

In [17]:
final_data_frame["is_prediction_correct"].mean()

0.9292035398230089

In [18]:
# Create a new DataFrame with incorrect predictions
incorrect_predictions = final_data_frame[~final_data_frame['is_prediction_correct']].copy()

# Create a column for the image name (assuming it's the last part of the image_path)
incorrect_predictions['image_name'] = incorrect_predictions['image_path'].apply(lambda x: x.split('/')[-1])

# Select and rename the columns we want
result = incorrect_predictions[['image_name', 'predicted', 'gt', 'word_label']]
result = result.rename(columns={'predicted': 'incorrect_response', 'gt': 'correct_response'})

# Combine the information into a single 'incorrect_responses' column
result['incorrect_responses'] = result.apply(lambda row: 
    f"Word: {row['word_label']}, Predicted: {row['incorrect_response']}, Correct: {row['correct_response']}", axis=1)

# Keep only the columns we need
final_result = result[['image_name', 'incorrect_responses']]

# Group by image_name to combine multiple incorrect responses for the same image
final_result = final_result.groupby('image_name')['incorrect_responses'].agg(lambda x: ' | '.join(x)).reset_index()

# Display the first few rows
print(final_result.head())

# Save to CSV (optional)
final_result.to_csv('incorrect_predictions.csv', index=False)

# Print the total number of incorrect predictions
print(f"\nTotal number of incorrect predictions: {len(final_result)}")

                              image_name  \
0  Acknowledgement_i11_m_s0_t0.3_fOS.png   
1  Acknowledgement_i11_m_s0_t0.4_fOS.png   
2  Acknowledgement_i11_m_s0_t0.5_fOS.png   
3   Acknowledgement_i13_n_s3_t0.4_fH.png   
4  Acknowledgement_i13_n_s3_t0.4_fOS.png   

                                 incorrect_responses  
0    Word: Acknowledgement, Predicted: a, Correct: m  
1  Word: Acknowledgement, Predicted: marker_not_f...  
2    Word: Acknowledgement, Predicted: e, Correct: m  
3    Word: Acknowledgement, Predicted: t, Correct: n  
4    Word: Acknowledgement, Predicted: t, Correct: n  

Total number of incorrect predictions: 88


In [19]:
import os
import shutil

# Create a new folder for incorrect images
incorrect_folder = "incorrect_images"
os.makedirs(incorrect_folder, exist_ok=True)

# Get the list of incorrect image names
incorrect_image_names = final_result['image_name'].tolist()

# Function to find the full path of an image
def find_image_path(image_name):
    for root, dirs, files in os.walk("."):
        if image_name in files:
            return os.path.join(root, image_name)
    return None

# Copy incorrect images to the new folder
for image_name in incorrect_image_names:
    source_path = find_image_path(image_name)
    if source_path:
        destination_path = os.path.join(incorrect_folder, image_name)
        shutil.copy2(source_path, destination_path)
        print(f"Copied {image_name} to {incorrect_folder}")
    else:
        print(f"Couldn't find {image_name}")

# Print summary
print(f"\nTotal images copied: {len(os.listdir(incorrect_folder))}")
print(f"Images are now in the '{incorrect_folder}' directory")

Copied Acknowledgement_i11_m_s0_t0.3_fOS.png to incorrect_images
Copied Acknowledgement_i11_m_s0_t0.4_fOS.png to incorrect_images
Copied Acknowledgement_i11_m_s0_t0.5_fOS.png to incorrect_images
Copied Acknowledgement_i13_n_s3_t0.4_fH.png to incorrect_images
Copied Acknowledgement_i13_n_s3_t0.4_fOS.png to incorrect_images
Copied Acknowledgement_i13_n_s3_t0.5_fH.png to incorrect_images
Copied Acknowledgement_i13_n_s3_t0.5_fOS.png to incorrect_images
Copied Subdermatoglyphic_i11_l_s2_t0.4_fH.png to incorrect_images
Copied Subdermatoglyphic_i11_l_s2_t0.5_fH.png to incorrect_images
Copied Subdermatoglyphic_i14_h_s0_t0.3_fH.png to incorrect_images
Copied Subdermatoglyphic_i14_h_s0_t0.5_fH.png to incorrect_images
Copied Subdermatoglyphic_i16_c_s1_t0.4_fOS.png to incorrect_images
Copied Subdermatoglyphic_i16_c_s2_t0.3_fH.png to incorrect_images
Copied Subdermatoglyphic_i16_c_s2_t0.4_fH.png to incorrect_images
Copied Subdermatoglyphic_i16_c_s3_t0.3_fH.png to incorrect_images
Copied Subdermatog